In [17]:
import pandas as pd
import numpy as np
import glob
import os

In [18]:
input_path = "/mnt/d/SEM_six_SGP/Crop_Price_V2/final"
output_path = "/mnt/d/SEM_six_SGP/Crop_Price_V2/processed"

os.makedirs(output_path, exist_ok=True)

files = glob.glob(os.path.join(input_path, "*.csv"))

print("Total files:", len(files))

Total files: 46


In [19]:
HORIZON = 4

for file in files:
    
    print("Processing:", file)
    
    df = pd.read_csv(file)
    
    # Sort properly
    df = df.sort_values(["district","commodity","year","month"])
    
    # Create datetime
    df["date"] = pd.to_datetime(
        df["year"].astype(str) + "-" + df["month"].astype(str) + "-01"
    )
    
    df = df.sort_values("date").reset_index(drop=True)
    
    
    # -----------------------------
    # TARGET (Harvest Horizon)
    # -----------------------------
    df["target_price"] = df["monthly_mean_price"].shift(-HORIZON)
    
    
    # -----------------------------
    # LAG FEATURES
    # -----------------------------
    lags = [1,2,3,6,12]
    
    for lag in lags:
        df[f"price_lag_{lag}"] = df["monthly_mean_price"].shift(lag)
    
    
    # -----------------------------
    # ROLLING FEATURES (Leakage Safe)
    # -----------------------------
    windows = [3,6]
    
    for window in windows:
        df[f"price_roll_mean_{window}"] = (
            df["monthly_mean_price"]
            .shift(1)
            .rolling(window)
            .mean()
        )
        
        df[f"price_roll_std_{window}"] = (
            df["monthly_mean_price"]
            .shift(1)
            .rolling(window)
            .std()
        )
    
    
    # -----------------------------
    # PRICE MOMENTUM
    # -----------------------------
    df["price_mom_1"] = (
        df["monthly_mean_price"] -
        df["monthly_mean_price"].shift(1)
    )
    
    df["price_mom_3"] = (
        df["monthly_mean_price"] -
        df["monthly_mean_price"].shift(3)
    )
    
    
    # -----------------------------
    # RAINFALL FEATURES
    # -----------------------------
    df["rain_roll_3"] = (
        df["monthly_rain_sum"]
        .shift(1)
        .rolling(3)
        .sum()
    )
    
    monthly_normal = df.groupby("month")["monthly_rain_sum"].transform("mean")
    df["rain_dev"] = df["monthly_rain_sum"] - monthly_normal
    
    
    # -----------------------------
    # NDVI FEATURES
    # -----------------------------
    df["ndvi_lag_1"] = df["monthly_ndvi_mean"].shift(1)
    
    df["ndvi_trend_3"] = (
        df["monthly_ndvi_mean"] -
        df["monthly_ndvi_mean"].shift(3)
    )
    
    
    # -----------------------------
    # SEASONAL ENCODING
    # -----------------------------
    df["month_sin"] = np.sin(2*np.pi*df["month"]/12)
    df["month_cos"] = np.cos(2*np.pi*df["month"]/12)
    
    
    # -----------------------------
    # DROP NA ROWS (created by lagging)
    # -----------------------------
    df = df.dropna().reset_index(drop=True)
    
    
    # -----------------------------
    # SAVE FILE
    # -----------------------------
    filename = os.path.basename(file)
    save_path = os.path.join(output_path, filename)
    
    df.to_csv(save_path, index=False)
    
    print("Saved:", save_path)

print("All files processed successfully ✅")

Processing: /mnt/d/SEM_six_SGP/Crop_Price_V2/final/ajwan_final.csv
Saved: /mnt/d/SEM_six_SGP/Crop_Price_V2/processed/ajwan_final.csv
Processing: /mnt/d/SEM_six_SGP/Crop_Price_V2/final/amaranthus_final.csv
Saved: /mnt/d/SEM_six_SGP/Crop_Price_V2/processed/amaranthus_final.csv
Processing: /mnt/d/SEM_six_SGP/Crop_Price_V2/final/apple_final.csv
Saved: /mnt/d/SEM_six_SGP/Crop_Price_V2/processed/apple_final.csv
Processing: /mnt/d/SEM_six_SGP/Crop_Price_V2/final/banana_final.csv
Saved: /mnt/d/SEM_six_SGP/Crop_Price_V2/processed/banana_final.csv
Processing: /mnt/d/SEM_six_SGP/Crop_Price_V2/final/beans_final.csv
Saved: /mnt/d/SEM_six_SGP/Crop_Price_V2/processed/beans_final.csv
Processing: /mnt/d/SEM_six_SGP/Crop_Price_V2/final/beetroot_final.csv
Saved: /mnt/d/SEM_six_SGP/Crop_Price_V2/processed/beetroot_final.csv
Processing: /mnt/d/SEM_six_SGP/Crop_Price_V2/final/brinjal_final.csv
Saved: /mnt/d/SEM_six_SGP/Crop_Price_V2/processed/brinjal_final.csv
Processing: /mnt/d/SEM_six_SGP/Crop_Price_V2/fi